In [8]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../data/bank_loan.db")
df = pd.read_sql("SELECT * FROM clean_data", conn)
conn.close()

print("Loaded:", df.shape)
df.head()

Loaded: (5000, 12)


,Age,Experience,Income,Family,CCAvg,Education,Mortgage,Personal Loan,Securities Account,CD Account,Online,CreditCard
0,25,1,49,4,0.0,1,0,0,1,0,0,0
1,45,19,34,3,0.0,1,0,0,1,0,0,0
2,39,15,11,1,0.0,1,0,0,0,0,0,0
3,35,9,100,1,0.0,2,0,0,0,0,0,0
4,35,8,45,4,0.0,2,0,0,0,0,0,1


In [9]:
edu_map = {1: "Undergrad", 2: "Graduate", 3: "Advanced"}

result = df.groupby("Education")["Income"].mean().round(1)
result.index = result.index.map(edu_map)

print("Avg Income by Education:")
print(result.to_string())

Avg Income by Education:
Education
Undergrad    85.6
Graduate     64.3
Advanced     66.1


In [10]:
result = df.groupby(["Education", "Family"])["Personal Loan"].mean().mul(100).round(1)
result.name = "Approval Rate (%)"

print("Loan Approval Rate by Education & Family Size:")
print(result.to_string())

Loan Approval Rate by Education & Family Size:
Education  Family
1          1          1.3
           2          0.6
           3         11.5
           4          9.7
2          1         12.3
           2         18.9
           3         11.5
           4         11.2
3          1         12.4
           2         13.9
           3         17.6
           4         12.1


In [11]:
print("Family Size Distribution:")
print(df["Family"].value_counts().sort_index())

print("\nEducation Distribution:")
edu_counts = df["Education"].value_counts().sort_index()
edu_counts.index = ["Undergrad", "Graduate", "Advanced"]
print(edu_counts)

Family Size Distribution:
Family
1    1472
2    1296
3    1010
4    1222
Name: count, dtype: int64

Education Distribution:
Undergrad    2096
Graduate     1403
Advanced     1501
Name: count, dtype: int64


In [12]:
pivot = df.pivot_table(
    values="Income",
    index="Education",
    columns="Family",
    aggfunc="mean"
).round(1)

pivot.index = ["Undergrad", "Graduate", "Advanced"]
pivot.columns = [f"Family {i}" for i in pivot.columns]

print("Average Income by Education & Family Size:")
pivot

Average Income by Education & Family Size:


,Family 1,Family 2,Family 3,Family 4
Undergrad,95.7,101.8,70.5,55.8
Graduate,64.4,69.4,61.7,63.4
Advanced,63.7,63.9,69.6,68.7


In [13]:
high_value = df[(df["Personal Loan"] == 1) & (df["Income"] > 100)]
high_value = high_value.sort_values("Income", ascending=False)

print(f"High income loan customers: {len(high_value)}")
high_value[["Age", "Income", "Education", "Family", "Mortgage", "CD Account"]].head(10)

High income loan customers: 438


,Age,Income,Education,Family,Mortgage,CD Account
2101,35,203,3,1,0,0
787,45,202,3,3,0,0
2337,43,201,2,1,0,0
4282,26,195,3,3,0,1
2956,62,195,3,4,522,1
2753,54,195,2,2,477,0
914,65,195,1,3,0,1
303,49,195,1,4,617,0
4267,52,194,2,2,0,0
47,37,194,3,4,211,1


In [14]:
corr = df.corr()["Personal Loan"].drop("Personal Loan").sort_values(ascending=False)

print("Correlation with Personal Loan:")
for col, val in corr.items():
    if pd.isna(val):
        continue
    bar = "█" * int(abs(val) * 50)
    sign = "+" if val > 0 else "-"
    print(f"{col:<22} {sign}{abs(val):.3f} {bar}")

Correlation with Personal Loan:
Income                 +0.502 █████████████████████████
CD Account             +0.316 ███████████████
Mortgage               +0.142 ███████
Education              +0.137 ██████
Family                 +0.061 ███
Securities Account     +0.022 █
Online                 +0.006 
CreditCard             +0.003 
Age                    -0.008 
Experience             -0.008 
